# Day 8: Numerical Fields and Missing Values Validation

## Objective
Validate the cleaned Superstore dataset by examining missing values, numerical data types, numerical ranges, invalid values, and potential outliers.

**Input:** `data/processed/cleaned_superstore.csv`

**Important interpretation rules:**
- `Row ID` is a numeric identifier, not a business measure.
- `Postal Code` is treated as an identifier and should remain text.
- Negative `Profit` values represent losses and are valid business observations.
- Zero `Discount` is valid.
- Outliers are flagged for review, not automatically removed.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_PATH = Path("data/processed/cleaned_superstore.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, dtype={"Postal Code": "string"})

print("Dataset loaded successfully")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Total cells:", df.size)
print("\nColumn data types:")
print(df.dtypes.to_string())


Dataset loaded successfully
Rows: 10194
Columns: 21
Total cells: 214074

Column data types:
Row ID              int64
Order ID              str
Order Date            str
Ship Date             str
Ship Mode             str
Customer ID           str
Customer Name         str
Segment               str
Country/Region        str
City                  str
State/Province        str
Postal Code        string
Region                str
Product ID            str
Category              str
Sub-Category          str
Product Name          str
Sales             float64
Quantity            int64
Discount          float64
Profit            float64


## 1. Missing-value analysis

In [2]:
missing_summary = pd.DataFrame({
    "Missing_Count": df.isna().sum(),
    "Missing_Percentage": (df.isna().mean() * 100).round(2),
    "Non_Missing_Count": df.notna().sum()
})

print("Missing-value summary:")
display(missing_summary)

total_missing = int(df.isna().sum().sum())
rows_with_missing = int(df.isna().any(axis=1).sum())

print("Total missing cells:", total_missing)
print("Columns containing missing values:", int((df.isna().sum() > 0).sum()))
print("Rows containing at least one missing value:", rows_with_missing)
print("Fully populated rows:", len(df) - rows_with_missing)


Missing-value summary:


,Missing_Count,Missing_Percentage,Non_Missing_Count
Row ID,0,0.0,10194
Order ID,0,0.0,10194
Order Date,0,0.0,10194
Ship Date,0,0.0,10194
Ship Mode,0,0.0,10194
Customer ID,0,0.0,10194
Customer Name,0,0.0,10194
Segment,0,0.0,10194
Country/Region,0,0.0,10194
City,0,0.0,10194


Total missing cells: 0
Columns containing missing values: 0
Rows containing at least one missing value: 0
Fully populated rows: 10194


## 2. Numerical-field summary

In [3]:
numeric_cols = ["Row ID", "Sales", "Quantity", "Discount", "Profit"]

numeric_summary = df[numeric_cols].describe().T
numeric_summary["Missing"] = df[numeric_cols].isna().sum()
numeric_summary["Zero_Count"] = (df[numeric_cols] == 0).sum()
numeric_summary["Negative_Count"] = (df[numeric_cols] < 0).sum()
numeric_summary["Positive_Count"] = (df[numeric_cols] > 0).sum()
numeric_summary["Infinite_Count"] = np.isinf(df[numeric_cols]).sum()

display(numeric_summary.round(3))


,count,mean,std,min,25%,50%,75%,max,Missing,Zero_Count,Negative_Count,Positive_Count,Infinite_Count
Row ID,10194.0,5097.500,2942.899,1.000,2549.250,5097.50,7645.750,10194.000,0,0,0,10194,0
Sales,10194.0,228.226,619.907,0.444,17.220,53.91,209.500,22638.480,0,0,0,10194,0
Quantity,10194.0,3.792,2.228,1.000,2.000,3.00,5.000,14.000,0,0,0,10194,0
Discount,10194.0,0.155,0.206,0.000,0.000,0.20,0.200,0.800,0,4925,0,5269,0
Profit,10194.0,28.673,232.465,-6599.978,1.761,8.69,29.298,8399.976,0,65,1901,8228,0


## 3. Field-specific validation

In [4]:
field_checks = {
    "Sales negative values": int((df["Sales"] < 0).sum()),
    "Sales zero values": int((df["Sales"] == 0).sum()),
    "Quantity negative values": int((df["Quantity"] < 0).sum()),
    "Quantity zero values": int((df["Quantity"] == 0).sum()),
    "Quantity non-integer values": int((~np.isclose(df["Quantity"] % 1, 0)).sum()),
    "Discount below 0": int((df["Discount"] < 0).sum()),
    "Discount above 1": int((df["Discount"] > 1).sum()),
    "Negative Profit records": int((df["Profit"] < 0).sum()),
    "Zero Profit records": int((df["Profit"] == 0).sum()),
    "Duplicate Row IDs": int(df["Row ID"].duplicated().sum()),
    "Infinite numerical values": int(np.isinf(df[numeric_cols]).sum().sum())
}

print("Field-specific validation:")
for check, count in field_checks.items():
    print(f"{check}: {count}")

print("\nNumerical ranges:")
for col in numeric_cols:
    print(f"{col}: minimum={df[col].min()}, maximum={df[col].max()}")


Field-specific validation:
Sales negative values: 0
Sales zero values: 0
Quantity negative values: 0
Quantity zero values: 0
Quantity non-integer values: 0
Discount below 0: 0
Discount above 1: 0
Negative Profit records: 1901
Zero Profit records: 65
Duplicate Row IDs: 0
Infinite numerical values: 0

Numerical ranges:
Row ID: minimum=1, maximum=10194
Sales: minimum=0.4439999999999999, maximum=22638.48
Quantity: minimum=1, maximum=14
Discount: minimum=0.0, maximum=0.8
Profit: minimum=-6599.978000000001, maximum=8399.975999999999


## 4. Data-type and identifier validation

In [5]:
identifier_cols = ["Order ID", "Customer ID", "Product ID", "Postal Code"]

print("Identifier field data types:")
for col in identifier_cols:
    print(f"{col}: {df[col].dtype}")

print("\nRow ID uniqueness:")
print("Unique Row IDs:", df["Row ID"].nunique())
print("Expected rows:", len(df))
print("Minimum Row ID:", df["Row ID"].min())
print("Maximum Row ID:", df["Row ID"].max())

row_id_sequence_valid = (
    df["Row ID"].is_unique
    and df["Row ID"].min() == 1
    and df["Row ID"].max() == len(df)
)
print("Row ID sequence valid:", row_id_sequence_valid)


Identifier field data types:
Order ID: str
Customer ID: str
Product ID: str
Postal Code: string

Row ID uniqueness:
Unique Row IDs: 10194
Expected rows: 10194
Minimum Row ID: 1
Maximum Row ID: 10194
Row ID sequence valid: True


## 5. Potential outlier identification

The IQR method is used to flag unusually low or high values in Sales, Quantity, and Profit.

These observations are **not automatically errors**. Large orders, high-value products, and substantial losses or profits may be legitimate business records.


In [6]:
outlier_rows = []

for col in ["Sales", "Quantity", "Profit"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    mask = (df[col] < lower_bound) | (df[col] > upper_bound)

    outlier_rows.append({
        "Field": col,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower_Bound": lower_bound,
        "Upper_Bound": upper_bound,
        "Outlier_Count": int(mask.sum()),
        "Outlier_Percentage": round(mask.mean() * 100, 2)
    })

outlier_summary = pd.DataFrame(outlier_rows).set_index("Field")
display(outlier_summary.round(3))

print("Outliers are flagged for investigation only; no rows are removed.")


,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count,Outlier_Percentage
Field,,,,,,,
Sales,17.220,209.500,192.280,-271.200,497.920,1183,11.60
Quantity,2.000,5.000,3.000,-2.500,9.500,175,1.72
Profit,1.761,29.298,27.537,-39.545,70.604,1913,18.77


Outliers are flagged for investigation only; no rows are removed.


## 6. Final validation checklist

In [7]:
validation_checks = {
    "Dataset contains 21 columns": df.shape[1] == 21,
    "No missing values": total_missing == 0,
    "All numerical values are finite": int(np.isinf(df[numeric_cols]).sum().sum()) == 0,
    "Sales values are non-negative": (df["Sales"] >= 0).all(),
    "Quantity values are positive": (df["Quantity"] > 0).all(),
    "Quantity values are integers": np.isclose(df["Quantity"] % 1, 0).all(),
    "Discount values are between 0 and 1": df["Discount"].between(0, 1).all(),
    "Row IDs are unique": df["Row ID"].is_unique,
    "Row IDs form a continuous sequence": row_id_sequence_valid,
    "Postal Code is stored as text": pd.api.types.is_string_dtype(df["Postal Code"])
}

validation_results = pd.DataFrame([
    {"Validation": name, "Result": "PASS" if passed else "FAIL"}
    for name, passed in validation_checks.items()
])

display(validation_results)

passed_count = sum(validation_checks.values())
total_checks = len(validation_checks)

print(f"Validation checks passed: {passed_count}/{total_checks}")

if passed_count != total_checks:
    print("Review failed checks before considering validation complete.")
else:
    print("All validation checks passed.")

print("\nNegative Profit records:", int((df["Profit"] < 0).sum()))
print("These are retained as valid loss records.")


,Validation,Result
0,Dataset contains 21 columns,PASS
1,No missing values,PASS
2,All numerical values are finite,PASS
3,Sales values are non-negative,PASS
4,Quantity values are positive,PASS
5,Quantity values are integers,PASS
6,Discount values are between 0 and 1,PASS
7,Row IDs are unique,PASS
8,Row IDs form a continuous sequence,PASS
9,Postal Code is stored as text,PASS


Validation checks passed: 10/10
All validation checks passed.

Negative Profit records: 1901
These are retained as valid loss records.


## 7. Conclusion

Complete this section after reviewing the executed results.

The analysis covers missing values, numerical distributions, field-specific rules, identifier data types, and potential outliers. Outlier observations are retained unless further investigation establishes that they are data errors.

Negative Profit and zero Discount are valid business cases and are not treated as data-quality failures.
